In [ ]:
# --- Colab bootstrap: in Colab, clone + install the repo; locally, do nothing ---
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/CKmellow/per3-genotype-clock-gene-timing.git"
BRANCH = "main"  # change to your branch name to test it in Colab
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    repo = Path("/content/per3-genotype-clock-gene-timing")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(repo)], check=True)
    os.chdir(repo)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
    src = str(repo / "src")
else:  # local: find the repo root (works from notebooks/ or notebooks/stages/)
    here = Path.cwd().resolve()
    root = next((p for p in [here, *here.parents] if (p / "pyproject.toml").exists()), here)
    src = str(root / "src")
if src not in sys.path:
    sys.path.insert(0, src)  # makes `import per3clock` work without a kernel restart

from per3clock import config  # noqa: E402

config.set_seeds()
config.ensure_dirs()
print("per3clock root:", config.ROOT, "| Colab:", IN_COLAB)

# Stage 5 – Core analysis: peak timing by PER3 genotype

## Minimum evidence (lecturer's brief)
*Brief, stage 5: "Apply the methods that directly answer your question." Minimum evidence: "Results table, key figures, statistical evidence" (example: "with FDR correction").*

- [ ] Methods that directly answer objectives 1–3 (per-subject peak times; 4/4 vs 5/5 comparison; melatonin-relative phase)
- [ ] **Results table** (estimates, effect sizes in hours with 95% CIs, test statistics, p-values)
- [ ] **Key figures** (e.g. acrophase rose plots, forest plot of shifts)
- [ ] **Statistical evidence** for each objective, with assumptions checked
- [ ] **FDR correction** (Benjamini–Hochberg) wherever many tests are run: the genome-wide rhythmicity scan, and across the primary + secondary genes
- [ ] Sensitivity analyses: `strict=True` sample set; Sleep Restriction; probe-mean (`collapse="mean"`) vs selected probe
- [ ] Short interpretation of your own results
- [ ] Report what the data does **and does not** show. A null, negative or inconclusive result is a valid result: report it with its uncertainty. With ~12 subjects per genotype, a non-significant shift is *inconclusive*, not proof of no shift. Give the CI.

## Contract
**Sample sets (shared defaults, `data/README.md`):**
- objectives 1–2 (clock time): `data.analysis_samples()` → 24 subjects (11 × 4/4, 13 × 5/5), 221 samples
- objective 3 (melatonin phase, `time_col="circadian_phase_h"`): `data.analysis_samples(for_phase_analysis=True)` → 23 subjects (AF0182 SE dropped)
- sensitivity: `strict=True` (20 subjects) and `condition="Sleep Restriction"`

**Inputs:** `data.load_clock_genes()` (one probe per gene; `collapse="mean"` for the probe sensitivity check), expression matrix via `data.collapse_probes_to_genes(data.load_expression_matrix())`, expressed genes from Stage 3 (`qc.expressed_genes`).

**Implemented in `per3clock.rhythm`:** `fit_subject_cosinors`, `rhythmicity_scan` (population-mean cosinor). **In `per3clock.circstats`:** `circular_mean_h`, `circular_diff_h`, `watson_williams_test`, `permutation_test_acrophase`, `bootstrap_ci_diff_h`, `genotype_shift_table`. **In `per3clock.plotting`:** `plot_acrophase_rose`, `plot_shift_forest`.

**Outputs:**
- `results/tables/acrophase_by_subject.csv`: subject × gene × condition × time base (clock time / circadian phase), primary sets
- `results/tables/acrophase_genotype_tests.csv`: gene, time base, condition, sample set, mean acrophase 4/4 and 5/5, shift (h, 5/5 − 4/4), 95% CI, permutation p, Watson–Williams p, BH q (primary analysis)
- `results/tables/acrophase_sensitivity_by_subject.csv`, `acrophase_sensitivity_tests.csv`: the same for the sensitivity analyses
- `results/tables/rhythmic_genes_SE.csv`: genome-wide scan with q-values (**input to Part 4**)
- figures: `acrophase_rose_PER2.png` / `_PER3` / `_NR1D1`, `acrophase_shift_forest.png`

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

from per3clock import circstats, config, data, plotting, qc, rhythm

plotting.set_style()
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
FIG_DPI = 300                    # report figures
GENES = config.GENES_PRIMARY     # PER2, PER3, NR1D1
N_PERM = N_BOOT = 10_000

## 5.1 Per-subject cosinor fits (objective 1)

One cosinor (24-h period) per subject × gene, fitted twice:
- **Clock time** (`clock_time_h`), on the main set (`analysis_samples()`, 24 subjects). Clock time wraps at midnight; the cosinor is periodic, so no unwrapping is needed.
- **Circadian phase** (`circadian_phase_h`, hours from the inferred melatonin midpoint), on the phase set (`analysis_samples(for_phase_analysis=True)`, 23 subjects: AF0182 SE has inconsistent phase labels).

Each subject is fitted on its own, so between-person level differences (Stage 4: ~77 % of the variance) do not enter the peak-time estimates. Every subject has ≥ 6 samples, so the default `min_points=6` removes nothing.

In [ ]:
meta = data.load_metadata()
main = data.analysis_samples(metadata=meta)                                # 24 subjects, 221 samples
phase_set = data.analysis_samples(metadata=meta, for_phase_analysis=True)  # 23 subjects, 214 samples
clock = data.load_clock_genes()                                            # one probe per gene

clock_fits = rhythm.fit_subject_cosinors(
    clock[clock.sample_id.isin(main.sample_id)], time_col="clock_time_h", genes=GENES)
phase_fits = rhythm.fit_subject_cosinors(
    clock[clock.sample_id.isin(phase_set.sample_id)], time_col="circadian_phase_h", genes=GENES)

acrophase_by_subject = pd.concat([
    clock_fits.assign(time_base="clock_time"),
    phase_fits.assign(time_base="circadian_phase"),
], ignore_index=True).assign(sample_set="Primary")
acrophase_by_subject.to_csv(config.TABLES_DIR / "acrophase_by_subject.csv", index=False)

# How many subjects have a clear rhythm of their own? A peak time from a flat, noisy
# series is mostly noise, so this tells us how much to trust each gene's comparison.
rhythmic_fits = (acrophase_by_subject
                 .assign(rhythmic=lambda d: d.p < 0.05)
                 .groupby(["gene", "time_base", "per3_genotype"])
                 .agg(subjects=("subject", "size"), rhythmic_p05=("rhythmic", "sum"),
                      median_amplitude=("amplitude", "median"), median_r2=("r2", "median")))
display(rhythmic_fits.round(3))

**Per-subject fits.** 141 fits (24 subjects × 3 genes on clock time, 23 × 3 on circadian phase). How many subjects have a significant rhythm of their own (p < 0.05) matters for how far each gene's comparison can be trusted:
- **NR1D1:** 5 of 11 (4/4) and 8 of 13 (5/5) subjects; median R² 0.63–0.73. The most reliable peak times.
- **PER2:** 2 of 11 and 7 of 13; median amplitude 0.10 (4/4) vs 0.20 (5/5) log2.
- **PER3:** 2 of 11 and 5 of 13; median R² 0.35–0.54. **Most PER3 peak times come from series without a detectable rhythm**, so they are close to noise (hollow points in the rose plot below). Any PER3 comparison will have a wide CI and cannot rule out large shifts in either direction.

Fewer 4/4 than 5/5 subjects reach p < 0.05 for every gene, with lower median amplitudes. This is descriptive only (no test here) and could be chance with 11 vs 13 subjects.

## 5.2 Genotype comparison in clock time (objective 2)

For each gene and time base, the per-subject peak times of the two genotypes are compared with `circstats.genotype_shift_table`:
- **shift** = circular mean (5/5) − circular mean (4/4) in hours; **negative = 5/5 earlier** (the hypothesis);
- **95 % CI** by bootstrap, resampling subjects within genotype;
- **permutation test** (10,000 label shuffles; no distributional assumption) as the primary test;
- **Watson–Williams test** as a parametric check. Its assumptions (von Mises, similar concentration in both groups, not too dispersed) are checked and any problem is listed in `ww_assumptions`; where they fail, the permutation p is the one to read;
- **Benjamini–Hochberg q** across the six primary comparisons (3 genes × 2 time bases), one family of tests.

`rbar` is the mean resultant length of each group's peak times: 1 = every subject peaks at the same time, 0 = no common peak.

In [ ]:
primary_tests = circstats.genotype_shift_table(
    acrophase_by_subject, by=("gene", "time_base"), n_perm=N_PERM, n_boot=N_BOOT)
primary_tests.insert(2, "condition", config.MAIN_CONDITION)
primary_tests.insert(3, "sample_set", "Primary")
primary_tests.to_csv(config.TABLES_DIR / "acrophase_genotype_tests.csv", index=False)

show = ["gene", "time_base", "n_44", "n_55", "mean_acrophase_44_h", "mean_acrophase_55_h",
        "rbar_44", "rbar_55", "shift_h", "ci_lower_h", "ci_upper_h", "permutation_p",
        "permutation_q", "watson_williams_p", "ww_assumptions"]
display(primary_tests.loc[primary_tests.time_base == "clock_time", show].round(3))

In [ ]:
for gene in GENES:
    fig = plotting.plot_acrophase_rose(acrophase_by_subject, gene)
    plotting.save_fig(fig, f"acrophase_rose_{gene}", dpi=FIG_DPI)

**Clock time (objective 2).** All three point estimates are negative (5/5 earlier), the direction of the hypothesis, but **every 95 % CI includes zero** and no comparison approaches significance (permutation p 0.24–0.66, BH q = 0.66 for all six primary comparisons):

| gene | shift 5/5 − 4/4 | 95 % CI | permutation p |
|---|---|---|---|
| NR1D1 | −1.16 h | −3.09 to +0.68 | 0.24 |
| PER2 | −0.51 h | −2.57 to +1.72 | 0.66 |
| PER3 | −1.36 h | −6.29 to +2.94 | 0.58 |

Watson–Williams agrees with the permutation test for every gene, and its assumptions held in all six primary comparisons (`ww_assumptions` empty). Within each genotype, NR1D1 and PER2 subjects peak close together (rbar 0.80–0.95), while PER3 peak times are scattered (rbar 0.47–0.69).

**What this shows:** with 11 vs 13 subjects the data cannot detect a shift of the size the CIs allow. For NR1D1 they are compatible with anything from 5/5 peaking ~3 h earlier to ~0.7 h later. For PER3 the interval spans ~9 h, so it says almost nothing. This is an **inconclusive** result, not evidence of no shift.

## 5.3 Melatonin-relative phase (objective 3)

The same comparison against circadian phase (rows `circadian_phase` of `acrophase_genotype_tests.csv`, same BH family). If 5/5 peaks earlier on the clock only because its melatonin rhythm is earlier, the shift should shrink towards zero here.

In [ ]:
display(primary_tests.loc[primary_tests.time_base == "circadian_phase", show].round(3))

# Side by side: does the shift shrink when time is measured from melatonin?
display(primary_tests.pivot(index="gene", columns="time_base",
                            values=["shift_h", "ci_lower_h", "ci_upper_h"]).round(2))

**Circadian phase (objective 3).** Measured on the `circadian_phase_h` axis, the shifts move towards zero or reverse: NR1D1 +0.37 h (−1.17 to +1.68), PER2 +1.06 h (−0.47 to +2.69), PER3 −1.18 h (−5.27 to +2.23); none is significant (q = 0.66).

**Important caveat about this axis.** In Sleep Extension, `circadian_phase_h` equals `hours_awake − 19.5 h` for **every** subject in the phase set (AF0182, the only exception, is already excluded). The phase labels therefore carry no subject-specific melatonin information beyond each person's wake time: in this condition the "melatonin-relative" comparison is in effect a **comparison relative to each person's wake time**. Read objective 3 accordingly. (In Sleep Restriction the offset is −19.5 or −22.5 h depending on the subject, so it does vary there.)

**Reading objectives 2 and 3 together.** The 5/5 participants woke on average ~1.2 h earlier (Stage 4, `eda_subject_timing.csv`). On clock time, NR1D1 and PER3 peak ~1.2–1.4 h earlier in 5/5; relative to wake time that difference largely disappears. This is consistent with the clock-gene peaks following each person's schedule rather than an additional genotype effect on top of it, but none of the differences is statistically distinguishable from zero.

## 5.4 Sensitivity analyses

The primary comparisons are repeated on:
- **Strict_SE**: the authors' stricter sample rule (`strict=True`, 20 subjects; 19 for circadian phase);
- **Sleep_Restriction**: the other sleep condition (`condition="Sleep Restriction"`);
- **Probe_Mean_SE**: the main set, with all probes of a gene averaged (`collapse="mean"`) instead of the selected probe.

Each sensitivity set is its own BH family (3 genes × 2 time bases), like the primary analysis.

In [ ]:
sensitivity_settings = {
    "Strict_SE": {"kw": {"strict": True}, "collapse": "max_mean"},
    "Sleep_Restriction": {"kw": {"condition": "Sleep Restriction"}, "collapse": "max_mean"},
    "Probe_Mean_SE": {"kw": {}, "collapse": "mean"},
}

sens_fits, sens_tests = [], []
for name, s in sensitivity_settings.items():
    genes_long = data.load_clock_genes(collapse=s["collapse"])
    fits = []
    for time_base, time_col, phase in (("clock_time", "clock_time_h", False),
                                       ("circadian_phase", "circadian_phase_h", True)):
        samples = data.analysis_samples(metadata=meta, for_phase_analysis=phase, **s["kw"])
        f = rhythm.fit_subject_cosinors(genes_long[genes_long.sample_id.isin(samples.sample_id)],
                                        time_col=time_col, genes=GENES)
        fits.append(f.assign(time_base=time_base))
    fits = pd.concat(fits, ignore_index=True).assign(sample_set=name)
    tests = circstats.genotype_shift_table(fits, by=("gene", "time_base"),
                                           n_perm=N_PERM, n_boot=N_BOOT)
    tests.insert(2, "condition", fits.sleep_condition.iloc[0])
    tests.insert(3, "sample_set", name)
    sens_fits.append(fits)
    sens_tests.append(tests)

sensitivity_by_subject = pd.concat(sens_fits, ignore_index=True)
sensitivity_tests = pd.concat(sens_tests, ignore_index=True)
sensitivity_by_subject.to_csv(config.TABLES_DIR / "acrophase_sensitivity_by_subject.csv", index=False)
sensitivity_tests.to_csv(config.TABLES_DIR / "acrophase_sensitivity_tests.csv", index=False)

display(sensitivity_tests[["sample_set", "condition"] + show].round(3))

In [ ]:
forest = pd.concat([primary_tests, sensitivity_tests], ignore_index=True)
forest["gene"] = pd.Categorical(forest["gene"], GENES)
forest["time_base"] = pd.Categorical(forest["time_base"], ["clock_time", "circadian_phase"])
forest["sample_set"] = pd.Categorical(forest["sample_set"],
                                      ["Primary", *sensitivity_settings])
forest = forest.sort_values(["time_base", "gene", "sample_set"])
fig = plotting.plot_shift_forest(forest)
plotting.save_fig(fig, "acrophase_shift_forest", dpi=FIG_DPI)

**Sensitivity (`acrophase_shift_forest`).** No sensitivity analysis changes the conclusion: every CI includes zero and every BH q ≥ 0.44.
- **Strict_SE** (20 subjects) gives nearly the same estimates as the primary analysis (clock time NR1D1 −0.94 h, PER2 −0.76 h, PER3 −0.87 h).
- **Probe_Mean_SE** gives the largest clock-time shifts: PER3 −2.00 h (95 % CI −4.41 to +0.37, p = 0.09) and PER2 −1.70 h (−4.38 to +0.94). Averaging the probes makes PER3's peak times more concentrated (rbar 0.70–0.93 vs 0.47–0.69). However, Stage 3 found the two extra PER3 probes near background, so this may be a less noisy or a more diluted measure; it is a sensitivity check, not the primary result. Still not significant (q = 0.44).
- **Sleep_Restriction:** point estimates flip to 5/5 later on both axes, with very wide CIs for PER3 (−8.5 to +6.2 h on clock time). The Watson–Williams assumptions fail for PER3 here (low and unequal concentration: the 5/5 peak times are almost uniformly spread, rbar 0.12–0.27); read the permutation p.

Across all 24 comparisons the CIs are consistent with no shift, and the direction of the point estimates is not stable across conditions.

## 5.5 Genome-wide rhythmicity scan

**Question:** which genes have a daily rhythm that is shared across people? This is the input to Part 4 (enrichment) and puts the clock genes in context.

**Method: population-mean cosinor** (`rhythm.rhythmicity_scan`; Nelson et al. 1979; Cornelissen 2014). Each subject gets its own cosinor for every gene, which also removes that person's expression level; a one-sample Hotelling T² test then asks whether the subjects' rhythm coefficients average to zero. Subjects are the unit (23 here), so repeated samples from one person are not treated as independent, and a gene counts as rhythmic only if people share a **consistent peak time**. Rhythms with random phases cancel out. Benjamini–Hochberg FDR across all genes.

**Inputs:** the 17,528 expressed genes from Stage 3 (`qc.expressed_genes`, the agreed feature set and enrichment background) on the phase set (23 subjects, AF0182 SE excluded because its phase labels are unreliable), against circadian phase, i.e. the internal clock rather than the sleep–wake schedule.

In [ ]:
expr = data.load_expression_matrix()
annot = data.load_probe_annotation()
expressed, threshold = qc.expressed_genes(expr, annot)   # Stage 3 feature set (main SE samples)
scan_expr = expressed[phase_set.sample_id]
print(f"{len(scan_expr):,} expressed genes x {scan_expr.shape[1]} samples "
      f"({phase_set.subject.nunique()} subjects)")

scan = rhythm.rhythmicity_scan(scan_expr, phase_set, time_col="circadian_phase_h")
scan.to_csv(config.TABLES_DIR / "rhythmic_genes_SE.csv")

n_sig = int((scan.q < 0.05).sum())
print(f"rhythmic at q < 0.05: {n_sig:,} of {len(scan):,} genes ({100 * n_sig / len(scan):.1f} %)")
display(scan.head(10).round(4))

# The clock genes in the genome-wide context
clock_rows = scan.loc[[g for g in config.CLOCK_GENES if g in scan.index]]
clock_rows = clock_rows.assign(rank=[int(scan.index.get_loc(g)) + 1 for g in clock_rows.index])
display(clock_rows.round(4))

**Scan result.** 9,577 of 17,528 expressed genes (55 %) have a rhythm shared across subjects at q < 0.05; 7,037 also have an amplitude ≥ 0.1 log2 (peak-to-trough ≥ 15 %) and 2,007 an amplitude ≥ 0.2. The clock genes fall where expected: NR1D1 (rank 129), NR1D2, PER2, PER1 and PER3 (q = 0.0008) are rhythmic; ARNTL, CRY1 and CLOCK are not (q 0.15–0.92), matching Stage 3/4.

**Why so many genes, and what it does not mean.**
- *Power, not effect size.* 23 subjects × ~9 samples detect very small but consistent rhythms; the median amplitude is only 0.10 log2. Significance here means "consistent across people", not "large".
- *Time awake.* In this protocol circadian phase and hours awake advance together (5.3), so changes driven by extended wakefulness that are shared across subjects can look like part of a 24-h rhythm. The two cannot be fully separated in this design.
- *Blood composition.* The peak times are strongly bimodal (most genes peak at phase ~21–23 h or ~8–11 h). The top genes (LEF1, LRRN3, NELL2, BACH2) are lymphocyte markers, so a large part of the scan probably reflects daily changes in the cell types circulating in blood rather than clocks inside each cell. Part 4 should keep this in mind when interpreting enrichment.

**For Part 4:** `rhythmic_genes_SE.csv` has the population rhythm (`amplitude`, `acrophase_h`), `p` and `q` for all 17,528 expressed genes, which are also the enrichment background. A foreground of q < 0.05 **and** amplitude ≥ 0.1 log2 is a reasonable choice; whichever is used, state it.

## Interpretation

- **Objective 1 (per-subject peak times).** NR1D1 and PER2 have clear rhythms in most subjects and well-defined peak times; PER3 is rhythmic at the group level (genome scan q = 0.0008) but only 7 of 24 subjects show a significant PER3 rhythm on their own, so individual PER3 peak times are imprecise.
- **Objective 2 (genotype, clock time).** 5/5 peaks earlier for all three genes (NR1D1 −1.2 h, PER2 −0.5 h, PER3 −1.4 h), but every 95 % CI includes zero (q = 0.66). The data do **not** show a genotype shift; they also cannot exclude shifts of a few hours (NR1D1: −3.1 to +0.7 h; PER3: −6.3 to +2.9 h).
- **Objective 3 (melatonin-relative).** On the circadian-phase axis the shifts shrink or reverse (NR1D1 +0.4 h, PER2 +1.1 h, PER3 −1.2 h; all CIs include zero). In Sleep Extension that axis is each subject's wake time minus a constant, so it is effectively a wake-relative comparison; the earlier clock-time peaks of 5/5 are about the size of their earlier wake times (~1.2 h, Stage 4).
- **Robustness.** The conclusion holds in all three sensitivity analyses (24 comparisons, all CIs include zero, q ≥ 0.44). The probe-mean analysis gives the largest PER3 shift (−2.0 h, p = 0.09).
- **Genome-wide.** Shared daily rhythms are widespread in blood (55 % of expressed genes at q < 0.05, mostly small amplitudes and probably partly cell-composition rhythms); the core clock genes behave as expected.
- **What the data do not show.** With 11 vs 13 subjects, 3-h sampling and binned phase labels, a non-significant result is **inconclusive**, not evidence that PER3 genotype has no effect on clock-gene timing. On clock time, a 5/5 peak 1–2 h earlier is within the 95 % CI for every gene.